In [1]:
import findspark 
findspark.init()

In [2]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [3]:
users = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/users.txt")
occupation = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/occupation.txt")
ratings1 = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/ratings_1.txt")
ratings2 = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/ratings_2.txt")
ratings = ratings1.union(ratings2)

In [4]:
# (UserID -> OccupationID)
users_rdd = users.map(lambda line: line.split(",")).map(lambda x: ((x[0]), (x[3])))
users_rdd.take(5)

[('1', '3'), ('2', '7'), ('3', '2'), ('4', '10'), ('5', '1')]

In [5]:
# (UserID → Rating)
ratings_rdd = ratings.map(lambda x: x.split(",")).map(lambda x: ((x[0]), float(x[2])))
ratings_rdd.take(5)

[('7', 4.5), ('23', 3.5), ('45', 4.0), ('12', 3.0), ('38', 4.5)]

In [6]:
# (OccupationID -> Name)
occupation_rdd = occupation.map(lambda x: x.split(",")).map(lambda x: ((x[0]), x[1]))
occupation_rdd.take(5)

[('1', 'Programmer'),
 ('2', 'Doctor'),
 ('3', 'Engineer'),
 ('4', 'Teacher'),
 ('5', 'Lawyer')]

#### Với mỗi rating, gán thông tin Occupation theo UserID.

In [8]:
# JOIN users_rdd + occupation_rdd -> (UserID -> Occupation)
user_occ = users_rdd.join(occupation_rdd)
user_occ.take(5)

[('4', ('10', 'Teacher')),
 ('10', ('14', 'Accountant')),
 ('12', ('8', 'Designer')),
 ('3', ('2', 'Engineer')),
 ('6', ('5', 'Artist'))]

In [9]:
user_occ = user_occ.map(lambda x: (x[0], x[1][1]))
user_occ.take(5)

[('4', 'Teacher'),
 ('10', 'Accountant'),
 ('12', 'Designer'),
 ('3', 'Engineer'),
 ('6', 'Artist')]

In [10]:
# Join ratings_rdd + user_occ -> (Occupation, Rating)
user_occ_rating = ratings_rdd.join(user_occ)
user_occ_rating.take(5)

[('12', (3.0, 'Designer')),
 ('12', (3.5, 'Designer')),
 ('12', (4.0, 'Designer')),
 ('12', (4.5, 'Designer')),
 ('4', (4.0, 'Teacher'))]

#### Phát hành cặp key-value với key là Occupation và value là (rating, 1).

In [12]:
occ_rating = user_occ_rating.map(lambda x: (x[1][1], (x[1][0], 1)))
occ_rating.take(5)

[('Designer', (3.0, 1)),
 ('Designer', (3.5, 1)),
 ('Designer', (4.0, 1)),
 ('Designer', (4.5, 1)),
 ('Teacher', (4.0, 1))]

#### Reduce để tính tổng điểm và số lượt cho mỗi Occupation, sau đó tính trung bình rating.

In [14]:
sum_count_rdd = occ_rating.reduceByKey(lambda a,b : (a[0] + b[0], a[1] + b[1]))
sum_count_rdd.take(5)

[('Manager', (21.0, 5)),
 ('Nurse', (28.5, 8)),
 ('Artist', (19.0, 6)),
 ('Designer', (15.0, 4)),
 ('Teacher', (4.0, 1))]

In [15]:
avg_rdd = sum_count_rdd.mapValues(lambda x: (x[0] / x[1], x[1]))
for row in avg_rdd.collect():
    print(row)

('Manager', (4.2, 5))
('Nurse', (3.5625, 8))
('Artist', (3.1666666666666665, 6))
('Designer', (3.75, 4))
('Teacher', (4.0, 1))
('Programmer', (4.0, 1))
('Consultant', (3.75, 2))
('Salesperson', (3.7, 5))
('Engineer', (3.6, 5))
('Student', (3.6, 5))
('Doctor', (3.75, 2))
('Researcher', (3.4, 5))
('Accountant', (4.0, 2))
('Journalist', (4.1, 5))
('Lawyer', (4.3, 5))
